# Capítulo 3 · Algoritmo de Simon

## Objetivos

1. Entender el problema de Simon y su estructura de periodicidad oculta.
2. Implementar el oráculo de Simon para cadenas de $n$ bits.
3. Verificar que el procedimiento cuántico + post-procesamiento clásico (álgebra lineal mod 2) recupera el período $\mathbf{s}$.

---

## 3C.1 El problema

Dada $f: \{0,1\}^n \to \{0,1\}^n$ con la promesa de que existe un vector oculto $\mathbf{s} \neq \mathbf{0}$ tal que:

$$f(\mathbf{x}) = f(\mathbf{y}) \iff \mathbf{y} = \mathbf{x} \oplus \mathbf{s}$$

El algoritmo de Simon encuentra $\mathbf{s}$ usando $O(n)$ llamadas al oráculo, frente a la solución clásica que necesita $O(2^{n/2})$ consultas.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


def simon_oracle(secret: str) -> QuantumCircuit:
    """Oráculo de Simon para el período oculto s.

    Implementa una función periódica basada en la cadena secreta s:
    copia x → y (CNOT cruzados) y XOR con s si la primera mitad coincide.

    Parámetros
    ----------
    secret : str
        Cadena binaria de longitud n (el período oculto).
    """
    n = len(secret)
    qc = QuantumCircuit(2 * n, name=f'Simon({secret})')

    # Copiar qubits de entrada al registro de salida
    for i in range(n):
        qc.cx(i, n + i)

    # Aplicar XOR con s en el primer qubit que sea '1'
    pivot = next((i for i, b in enumerate(secret) if b == '1'), None)
    if pivot is not None:
        for i, bit in enumerate(secret):
            if bit == '1':
                qc.cx(pivot, n + i)
    return qc


def simon_circuit(secret: str) -> QuantumCircuit:
    """Circuito cuántico del algoritmo de Simon."""
    n = len(secret)
    qc = QuantumCircuit(2 * n, n)

    # Hadamard en qubits de entrada
    qc.h(range(n))
    qc.barrier()

    # Oráculo
    oracle = simon_oracle(secret)
    qc.compose(oracle, inplace=True)
    qc.barrier()

    # Segundo Hadamard en qubits de entrada
    qc.h(range(n))
    qc.barrier()

    # Medir sólo los qubits de entrada
    qc.measure(range(n), range(n))
    return qc


def simon_postprocessing(equations: list, n: int) -> str:
    """Resuelve el sistema lineal mod 2 para encontrar el período s.

    Etapas:
    1. Formamos una matriz A cuyas filas son los vectores y medidos.
    2. Aplicamos eliminación gaussiana mod 2 para encontrar el núcleo.
    """
    # Convertir ecuaciones a matriz binaria
    rows = []
    for eq in equations:
        row = [int(b) for b in eq]
        rows.append(row)

    A = np.array(rows, dtype=int) % 2

    # Eliminación gaussiana mod 2
    pivot_cols = []
    pivot_row = 0
    for col in range(n):
        found = None
        for row in range(pivot_row, len(A)):
            if A[row, col] == 1:
                found = row
                break
        if found is None:
            continue
        A[[pivot_row, found]] = A[[found, pivot_row]]
        pivot_cols.append(col)
        for row in range(len(A)):
            if row != pivot_row and A[row, col] == 1:
                A[row] = (A[row] + A[pivot_row]) % 2
        pivot_row += 1

    # Si hay más columnas libres, elegir s en el espacio nulo
    free_cols = [c for c in range(n) if c not in pivot_cols]
    if not free_cols:
        return '0' * n  # s = 0 implica función 1-a-1

    # Reconstruir s usando la primera columna libre
    s = np.zeros(n, dtype=int)
    s[free_cols[0]] = 1
    for i, col in enumerate(pivot_cols):
        if i < len(A) and A[i, free_cols[0]] == 1:
            s[col] = 1

    return ''.join(str(b) for b in s)


print('Funciones de Simon definidas.')

# Demo
secret = '110'
n = len(secret)
backend = AerSimulator()
qc_simon = simon_circuit(secret)
print(qc_simon.draw('text'))

In [ ]:
# Ejecutar varias veces para recolectar ecuaciones lineales
secret = '1101'
n = len(secret)
qc_simon = simon_circuit(secret)

# Necesitamos al menos n-1 ecuaciones linealmente independientes
backend = AerSimulator()
job = backend.run(qc_simon, shots=50)
counts = job.result().get_counts()

# Filtrar el resultado todo-ceros (no informativo)
equations = [eq[::-1] for eq in counts.keys() if eq != '0' * n]

print(f'Período secreto: s = {secret}')
print(f'Ecuaciones recolectadas (y·s = 0 mod 2):')
for eq in equations[:8]:
    dot = sum(int(a)*int(b) for a,b in zip(eq, secret)) % 2
    print(f'  y = {eq}, y·s = {dot}')

s_recovered = simon_postprocessing(equations, n)
print(f'\nPeríodo recuperado: s = {s_recovered}')
print(f'Correcto: {s_recovered == secret}')

## 3C.2 Ejercicios propuestos

1. ¿Cuántas mediciones cuánticas son suficientes en promedio para obtener $n-1$ ecuaciones linealmente independientes? Analiza la probabilidad de fallo tras $k$ medidas.

2. Implementa el oráculo de Simon para el caso $\mathbf{s} = \mathbf{0}$ (función inyectiva) y verifica que el algoritmo devuelve el vector cero.

3. ¿Puede el algoritmo de Simon resolver el problema del logaritmo discreto? Investiga las conexiones.